# 1. native attention
$$ Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V $$

$$softmax(z_i) = \frac{exp(z_i)}{\sum^K_{j=1}exp(z_j)}$$
这个公式表示有K个实数的向量$Z=[z_1, z_2, ... , z_k]$

In [1]:
import torch
import torch.nn.functional as F 

z = torch.tensor([2.0, 1.0, 0.1])
similarity = F.softmax(z, dim=0)
print(similarity)
print(sum(similarity))

tensor([0.6590, 0.2424, 0.0986])
tensor(1.0000)


标准 Attention 的计算流程：
GPU SRAM (fast)  ←→  HBM (slow)
    
    读取 Q, K        [从 HBM 读取]
       ↓
    计算 S           [写回 HBM] ← 第一次写入
       ↓
    读取 S           [从 HBM 读取]
       ↓
    计算 P           [写回 HBM] ← 第二次写入
       ↓
    读取 P, V        [从 HBM 读取]
       ↓
    计算 O           [写回 HBM]

缺陷：
- 长序列容易oom，如d_model为1024，那么Q@K需要存储1024 * 1024个数据，同时考虑batch和head_num，所需的显存更大
- 显存访问速度较SRAM慢（memory bound）


# 2. online safe softmax
